# PyTorch Tutorial 50: Federated Learning Basics

**Author:** PyTorch Tutorial Series | **Date:** 2026 | **Time:** ~1.5 hours

**Prerequisites:** Notebooks 00-05 (basic PyTorch, tensors, training loops)

---

## What You'll Learn

1. **The privacy problem** — why we can't always share data
2. **Federated Learning** — training together without sharing data
3. **FedAvg from scratch** — the simplest FL algorithm
4. **Non-IID data** — what happens when each client has different data
5. **Differential Privacy** — adding controlled noise for extra protection

This is the final notebook in the Edge ML series (Notebooks 46-50)!

## Section 1: The Privacy Problem

Imagine three hospitals. Each has thousands of patient X-rays. They all want to
build an AI that detects pneumonia. The problem? **They can't share patient data.**

- **HIPAA** (US healthcare law) forbids sharing patient records without consent.
- **GDPR** (EU law) restricts how personal data moves across borders.

So each hospital trains alone on its small dataset. The models are mediocre.

**What if each hospital trained locally and only shared what it *learned*?**

That's **federated learning** -- training together without sharing raw data.

### Real-World Examples

- **Google Gboard:** Your phone trains a next-word model locally. Google gets model updates, not your messages.
- **Apple Health:** On-device ML for health features. Your data stays on your phone.
- **Hospital networks:** Multiple hospitals train diagnostic models without exchanging patient records.

## Section 2: How Federated Learning Works

```
     SERVER (global model)
     /     |     \
 Client A  Client B  Client C  (each has private data)
```

**Each round:**
1. Server sends the current model to all clients.
2. Each client trains on its **own local data** for a few epochs.
3. Clients send **updated weights** back to the server.
4. Server **averages** all the weights together.
5. Repeat until convergence.

The raw data **NEVER** leaves the client. Only model weights travel.

## Section 3: Setup

Let's build federated learning from scratch using only PyTorch and MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import copy
import random
import numpy as np

# Reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

In [ ]:
class SimpleCNN(nn.Module):
    """A small CNN for MNIST. Same architecture across all clients."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
        x = x.view(-1, 32 * 7 * 7)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

print("Model parameters:", sum(p.numel() for p in SimpleCNN().parameters()))

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    "./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    "./data", train=False, download=True, transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
def split_iid(dataset, num_clients):
    """Split dataset into equal random chunks -- each client gets a similar mix."""
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    chunk_size = len(indices) // num_clients
    return [
        indices[i * chunk_size : (i + 1) * chunk_size]
        for i in range(num_clients)
    ]


def split_non_iid(dataset, num_clients, classes_per_client=2):
    """Each client gets only a few digit classes (non-IID).

    For example, Client 0 might only have digits 0 and 1,
    Client 1 might only have digits 2 and 3, etc.
    """
    # Group indices by label
    label_indices = {i: [] for i in range(10)}
    for idx in range(len(dataset)):
        label = int(dataset.targets[idx])
        label_indices[label].append(idx)

    # Assign classes round-robin to clients
    all_classes = list(range(10))
    client_indices = [[] for _ in range(num_clients)]
    for i in range(num_clients):
        assigned = [
            all_classes[(i * classes_per_client + j) % 10]
            for j in range(classes_per_client)
        ]
        for c in assigned:
            client_indices[i].extend(label_indices[c])
        random.shuffle(client_indices[i])

    return client_indices


print("Data splitting functions ready.")

## Section 4: FedAvg From Scratch

**FedAvg** (Federated Averaging) is the simplest federated learning algorithm.
Published by McMahan et al. in 2017, it is still the baseline everyone compares against.

The idea is dead simple:
1. Each client trains for a few epochs on local data.
2. The server **averages** all the client models together.
3. That averaged model becomes the new starting point.

Let's implement it.

In [ ]:
def federated_average(global_model, client_models):
    """Average model weights from all clients into the global model.

    This is the core of FedAvg: simple parameter-wise averaging.
    Each client contributes equally (unweighted average).
    """
    global_state = global_model.state_dict()
    for key in global_state:
        # Stack all client params and take the mean
        stacked = torch.stack(
            [client.state_dict()[key].float() for client in client_models]
        )
        global_state[key] = stacked.mean(dim=0)
    global_model.load_state_dict(global_state)
    return global_model


print("federated_average() ready -- just 10 lines of actual logic!")

In [ ]:
def client_train(model, train_loader, local_epochs=2, lr=0.01):
    """Train a model on one client's local data.

    Each client runs SGD for a few epochs on its private dataset.
    The updated model is returned (not the data).
    """
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(local_epochs):
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()
    return model


print("client_train() ready.")

In [ ]:
def evaluate_model(model, data_loader):
    """Check model accuracy on the test set."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


print("evaluate_model() ready.")

In [ ]:
def run_federated_learning(
    num_clients=5, num_rounds=10, local_epochs=2, client_indices=None
):
    """Full federated learning loop.

    Each round: distribute global model -> local training -> average.
    Returns list of accuracies per round for analysis.
    """
    global_model = SimpleCNN().to(DEVICE)
    accuracies = []

    for rnd in range(1, num_rounds + 1):
        client_models = []
        for i in range(num_clients):
            # Each client gets a fresh copy of the global model
            local_model = copy.deepcopy(global_model)
            loader = DataLoader(
                Subset(train_dataset, client_indices[i]),
                batch_size=64, shuffle=True
            )
            trained = client_train(local_model, loader, local_epochs)
            client_models.append(trained)

        # Server averages all client models
        global_model = federated_average(global_model, client_models)
        acc = evaluate_model(global_model, test_loader)
        accuracies.append(acc)
        print(f"  Round {rnd:2d}/{num_rounds} -- Accuracy: {acc:.4f}")

    return accuracies


print("Federated learning loop ready.")

In [ ]:
# Run federated learning with IID data split
print("=" * 50)
print("Running FedAvg with 5 clients (IID split)")
print("=" * 50)

iid_indices = split_iid(train_dataset, num_clients=5)
for i, idx in enumerate(iid_indices):
    print(f"  Client {i}: {len(idx)} samples")

iid_accuracies = run_federated_learning(
    num_clients=5, num_rounds=10, local_epochs=2,
    client_indices=iid_indices
)

print(f"\nFinal accuracy (IID): {iid_accuracies[-1]:.4f}")
print("Each client's data stayed completely private!")

## Section 5: IID vs Non-IID Data

In the real world, data is rarely distributed evenly.

- **IID** (Independent and Identically Distributed): Each client has a random
  mix of all classes. Like shuffling a deck and dealing cards.
- **Non-IID**: Each client has a biased subset. Hospital A only sees children,
  Hospital B only sees elderly patients.

Non-IID is the harder (and more realistic) scenario. Let's see what happens.

In [ ]:
# Run with non-IID split: each client gets only 2 digit classes
print("=" * 50)
print("Running FedAvg with 5 clients (Non-IID split)")
print("=" * 50)

non_iid_indices = split_non_iid(
    train_dataset, num_clients=5, classes_per_client=2
)
for i, idx in enumerate(non_iid_indices):
    unique = sorted(set(int(train_dataset.targets[j]) for j in idx))
    print(f"  Client {i}: {len(idx)} samples, classes: {unique}")

non_iid_accuracies = run_federated_learning(
    num_clients=5, num_rounds=10, local_epochs=2,
    client_indices=non_iid_indices
)

print(f"\nFinal accuracy (Non-IID): {non_iid_accuracies[-1]:.4f}")

In [ ]:
# Compare IID vs Non-IID convergence
print("\n" + "=" * 55)
print(f"{'Round':<8} {'IID Accuracy':<16} {'Non-IID Accuracy':<16}")
print("-" * 55)
for i in range(len(iid_accuracies)):
    print(
        f"{i+1:<8} {iid_accuracies[i]:<16.4f} "
        f"{non_iid_accuracies[i]:<16.4f}"
    )
print("=" * 55)
print("\nNon-IID is harder but FL still works -- just converges slower.")
print("In practice, techniques like FedProx help close this gap.")

## Section 6: Differential Privacy

Federated learning keeps raw data local. But there is a subtlety:
**model weights can leak information about the training data.**

An attacker who sees the weight updates could potentially reconstruct
training examples (this is called a "model inversion attack").

**Differential Privacy (DP)** is the solution: add carefully calibrated noise
so that no single training example significantly affects the output.

### How DP-SGD works:
1. **Clip gradients** -- cap the influence of any single example.
2. **Add noise** -- inject Gaussian noise proportional to the clip norm.
3. **Epsilon** controls the privacy-accuracy tradeoff:
   - Lower epsilon = more noise = stronger privacy = slightly less accurate.
   - Higher epsilon = less noise = weaker privacy = more accurate.

Let's implement it manually -- no external libraries needed.

In [ ]:
def client_train_dp(model, train_loader, local_epochs=2, lr=0.01,
                     max_grad_norm=1.0, noise_multiplier=0.5):
    """Train with Differential Privacy (DP-SGD).

    Two key additions over regular training:
    - Gradient clipping: limits each sample's influence.
    - Noise injection: adds Gaussian noise to clipped gradients.
    """
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    for _ in range(local_epochs):
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()

            # Step 1: Clip gradients to limit per-sample influence
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_grad_norm
            )

            # Step 2: Add calibrated Gaussian noise
            for param in model.parameters():
                if param.grad is not None:
                    noise = torch.randn_like(param.grad)
                    noise *= noise_multiplier * max_grad_norm
                    param.grad = param.grad + noise

            optimizer.step()
    return model


print("DP-SGD training function ready.")

In [ ]:
def run_fl_with_dp(num_clients=5, num_rounds=10, local_epochs=2,
                    noise_multiplier=0.5, client_indices=None):
    """Federated learning with differential privacy."""
    global_model = SimpleCNN().to(DEVICE)
    accuracies = []

    for rnd in range(1, num_rounds + 1):
        client_models = []
        for i in range(num_clients):
            local_model = copy.deepcopy(global_model)
            loader = DataLoader(
                Subset(train_dataset, client_indices[i]),
                batch_size=64, shuffle=True
            )
            trained = client_train_dp(
                local_model, loader, local_epochs,
                noise_multiplier=noise_multiplier
            )
            client_models.append(trained)

        global_model = federated_average(global_model, client_models)
        acc = evaluate_model(global_model, test_loader)
        accuracies.append(acc)
        print(f"  Round {rnd:2d}/{num_rounds} -- Accuracy: {acc:.4f}")

    return accuracies


print("FL+DP training loop ready.")

In [ ]:
# Run FL with different noise levels
iid_idx = split_iid(train_dataset, num_clients=5)

print("=" * 50)
print("FL + DP: noise_multiplier = 0.3 (mild privacy)")
print("=" * 50)
dp_mild = run_fl_with_dp(
    num_clients=5, num_rounds=10, noise_multiplier=0.3,
    client_indices=iid_idx
)

print("\n" + "=" * 50)
print("FL + DP: noise_multiplier = 1.0 (strong privacy)")
print("=" * 50)
dp_strong = run_fl_with_dp(
    num_clients=5, num_rounds=10, noise_multiplier=1.0,
    client_indices=iid_idx
)

In [ ]:
# Compare: No DP vs Mild DP vs Strong DP
print("\nPrivacy-Accuracy Tradeoff")
print("=" * 65)
header = f"{'Round':<8} {'No DP':<14} {'Mild DP (0.3)':<16} {'Strong DP (1.0)':<16}"
print(header)
print("-" * 65)
for i in range(len(iid_accuracies)):
    print(
        f"{i+1:<8} {iid_accuracies[i]:<14.4f} "
        f"{dp_mild[i]:<16.4f} {dp_strong[i]:<16.4f}"
    )
print("=" * 65)
print("\nMore noise = stronger privacy guarantee, but slightly lower accuracy.")
print("This is the fundamental privacy-accuracy tradeoff.")

## Section 7: The Flower Framework

We built FL from scratch to understand it. In production, use
**Flower** -- the most popular FL framework (2026).

```python
import flwr as fl

class MnistClient(fl.client.NumPyClient):
    def get_parameters(self):
        return [val.numpy() for val in model.state_dict().values()]

    def fit(self, parameters, config):
        set_parameters(model, parameters)
        train(model, train_loader, epochs=1)
        return self.get_parameters(), len(train_loader.dataset), {}

    def evaluate(self, parameters, config):
        set_parameters(model, parameters)
        loss, acc = test(model, test_loader)
        return loss, len(test_loader.dataset), {"accuracy": acc}

fl.client.start_numpy_client(server_address="localhost:8080",
                              client=MnistClient())

# Server side
strategy = fl.server.strategy.FedAvg(min_fit_clients=5)
fl.server.start_server(strategy=strategy, config={"num_rounds": 10})
```

**Flower supports:** FedAvg, FedProx, FedOpt, FedAdam and works with
PyTorch, TensorFlow, and JAX. We built it from scratch to learn;
**in practice, use Flower.**

## Section 8: When to Use What

| Scenario | Recommendation |
|---|---|
| Can share data freely | Regular centralized training (simplest, best accuracy) |
| Multiple data owners, moderate privacy | Federated Learning |
| Strict regulations (healthcare, finance) | FL + Differential Privacy |
| Maximum privacy guarantee | FL + DP + Secure Aggregation |

**Rules of thumb:**
- If you CAN centralize data, do it. Centralized training is simpler and more accurate.
- If you CAN'T share data (legal, ethical, practical reasons), use FL.
- If even the model weights could leak info, add Differential Privacy.
- For the highest security, add Secure Aggregation (encrypts weights in transit).

## Section 9: Try It Yourself

**Exercise 1:** Change to 10 clients instead of 5. Does FL take more rounds?

**Exercise 2:** Try `split_non_iid(dataset, 10, classes_per_client=1)`. How bad does it get?

**Exercise 3:** Implement **FedProx** -- add a proximal term to the client loss:

```python
proximal_term = 0.0
for local_p, global_p in zip(model.parameters(), global_model.parameters()):
    proximal_term += (local_p - global_p).norm(2)
loss = loss + (mu / 2) * proximal_term  # try mu = 0.01
```

Does it help with non-IID data?

## Section 10: Recap

What we learned:

- **The privacy problem:** Data often can't be shared (HIPAA, GDPR). FL lets us collaborate anyway.
- **Federated Learning:** Train locally, share only model weights. Raw data never leaves the client.
- **FedAvg:** Average the weights. Surprisingly effective and simple to implement.
- **Non-IID challenge:** Different data distributions slow convergence, but FL still works.
- **Differential Privacy:** Add calibrated noise so no single example can be recovered.

---

### This completes the Edge ML series (Notebooks 46-50)!

| Notebook | Topic |
|---|---|
| 46 | Edge ML Fundamentals -- running models on constrained devices |
| 47 | Making Models Smaller -- pruning, quantization, distillation |
| 48 | Deploying to Edge -- ONNX, TFLite, mobile/embedded targets |
| 49 | On-Device LLMs -- running language models on phones and laptops |
| 50 | Federated Learning -- privacy-preserving collaborative training |

From fundamentals to compression to deployment to privacy -- you now have
a complete toolkit for ML at the edge. Happy building!